In [1]:
# /Users/souravm/Documents/Projects/statement_parser/input/statement_.pdf

import os
import pandas as pd
import pdfplumber
import contextlib
import io
import sys
from itertools import chain
import warnings
warnings.filterwarnings("ignore")

# Input file path
input_file = r'input/Acct Statement_XX4725_29092024.pdf'
input_file = os.path.abspath(input_file)

# Only suppress stderr (which is where the CropBox message appears)
with contextlib.redirect_stderr(io.StringIO()):
    with pdfplumber.open(input_file) as pdf:

        # Extracting the table from the pages
        combined_data = []
        for i in range(len(pdf.pages)):
            print('Reading page:', i)
            page = pdf.pages[i]
            table = page.extract_table()
            if table is not None:
                combined_data = list(chain(combined_data, table))

# Removal of header and footer
conditions = ['OPENING BALANCE', 'transaction total', 'closing balance']

print('Filtering out header and footer')
filtered_list = [
    items for items in combined_data
    if not any(
        isinstance(item, str) and any(cond.lower() in item.lower() for cond in conditions)
        for item in items
    )
]
# Cleaning of \n
cleaned_list = [
    [item.replace('\n', ' ') if isinstance(item, str) else item for item in sublist]
    for sublist in filtered_list
]


Reading page: 0
Reading page: 1
Reading page: 2
Filtering out header and footer


In [2]:
print(combined_data)

[['Date', 'Narration', 'Chq./Ref.No.', 'ValueDt', 'WithdrawalAmt.', 'DepositAmt.', 'ClosingBalance'], ['30/08/24\n30/08/24\n31/08/24\n31/08/24\n02/09/24\n02/09/24\n03/09/24\n03/09/24\n03/09/24\n04/09/24\n05/09/24\n06/09/24\n07/09/24\n10/09/24\n10/09/24\n15/09/24', '082024SALARYCREDITFOR20532356/SOURAV\nIMPS-424320347279-SOURAVAXIS-UTIB-XXXXXX\nXXXXX7756-TRANSFER\n.IMPSP2P423409352210#21/08/2024210824\n-MIR2524395700585\nNEFTDR-UTIB0001355-SOURAVAXIS-NETBANK,\nMUM-N244243236619002-SEPTEXP\nNEFTDR-PUNB0017720-RICHAHORE-NETBANK,\nMUM-N246243238840545-TRANSFERRIC\nMEDCSI435584XXXXXX8577GOOGLEPLAYSE\nRCYBSS\n.IMPSP2P423714320862#24/08/2024240824\n-MIR2524598746529\n.IMPSP2P423821370132#25/08/2024250824\n-MIR2524699696806\n.IMPSP2P423921329460#26/08/2024260824\n-MIR2524600400182\nATW-435584XXXXXX8577-P3ENBL81-BANGALORE\nNEFTDR-PUNB0017720-RICHAHORE-NETBANK,\nMUM-N249243246520383-SHOES\n.IMPSP2P424320347279#30/08/2024300824\n-MIR2525013049515\nEMI148443967CHQS14844396700810924148\n443967\nNEF

In [169]:
# Input file path
input_file = r'input/Acct Statement_XX4725_29092024.pdf'
# input_file = r'input/statement_.pdf'
input_file = os.path.abspath(input_file)

print('Reading PDF file:', input_file)

table_settings = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "text",
    "intersection_tolerance": 15,
    "join_tolerance": 20,
}
with contextlib.redirect_stderr(io.StringIO()):
    with pdfplumber.open(input_file) as pdf:

        # Extracting the table from the pages
        combined_data = []
        for i in range(len(pdf.pages)):
            print('Reading page:', i)
            page = pdf.pages[i]
            table = page.extract_table(table_settings=table_settings)
            if table is not None:
                combined_data = list(chain(combined_data, table))
print('Extracted data:', combined_data)

# Removal of header and footer
conditions = ['OPENING BALANCE', 'transaction total', 'closing balance', 'Statementof account', 'From']

print('Filtering out header and footer')
filtered_list = [
    items for items in combined_data
    if not any(
        isinstance(item, str) and any(cond.lower() in item.lower() for cond in conditions)
        for item in items
    )
]

# Filter out sublists where all elements are empty strings
filtered_data = [row for row in filtered_list if not all(elem == '' for elem in row)]

# Cleaning of \n
cleaned_list = [
    [item.replace('\n', ' ') if isinstance(item, str) else item for item in sublist]
    for sublist in filtered_data
]

cleaned_list = [sublist for sublist in cleaned_list if None not in sublist]

# Find the index of the sublist containing 'STATEMENTSUMMARY'
index_to_cut = None
for i, sublist in enumerate(cleaned_list):
    if any('STATEMENTSUMMARY' in str(item) for item in sublist):
        index_to_cut = i
        break

# If found, slice the list to keep only elements before that index
if index_to_cut is not None:
    cleaned_list = cleaned_list[:index_to_cut]


Reading PDF file: g:\Coding Project\Expense_Analyzer\input\Acct Statement_XX4725_29092024.pdf
Reading page: 0
Reading page: 1
Reading page: 2
Extracted data: [['From : 29/08/2024 To : 29/09/2024 Statementof account', None, None, None, None, None, None], ['', '', '', '', '', '', ''], ['Date', 'Narration', 'Chq./Ref.No.', 'ValueDt', 'WithdrawalAmt.', 'DepositAmt.', 'ClosingBalance'], ['', '', '', '', '', '', ''], ['30/08/24', '082024SALARYCREDITFOR20532356/SOURAV', '0000408297529599', '30/08/24', '', '74,941.00', '102,004.42'], ['', '', '', '', '', '', ''], ['30/08/24', 'IMPS-424320347279-SOURAVAXIS-UTIB-XXXXXX', '0000424320347279', '30/08/24', '1,000.00', '', '101,004.42'], ['', '', '', '', '', '', ''], ['', 'XXXXX7756-TRANSFER', '', '', '', '', ''], ['', '', '', '', '', '', ''], ['31/08/24', '.IMPSP2P423409352210#21/08/2024210824', 'MIR2524395700585', '31/08/24', '4.14', '', '101,000.28'], ['', '', '', '', '', '', ''], ['', '-MIR2524395700585', '', '', '', '', ''], ['', '', '', '', '',

In [170]:
clean_hdfc_data = pd.DataFrame(cleaned_list)

In [171]:
cleaned_list = cleaned_list[1:]

index = 0
while index < (len(cleaned_list)-1):
# for index in range(len(cleaned_list)):
    item = cleaned_list[index][1]
    if len(cleaned_list[index][1])>len(cleaned_list[index+1][1]):
        new_particular = cleaned_list[index][1]+cleaned_list[index+1][1]
        cleaned_list[index][1]=new_particular
        index+=2
    else:
        index+=1


In [172]:
# cleaned_list = [sublist for sublist in cleaned_list if '' not in sublist[0]]

cleaned_list = [sublist for sublist in cleaned_list if sublist[0] != '']




In [173]:
cleaned_list

[['30/08/24',
  '082024SALARYCREDITFOR20532356/SOURAV',
  '0000408297529599',
  '30/08/24',
  '',
  '74,941.00',
  '102,004.42'],
 ['30/08/24',
  'IMPS-424320347279-SOURAVAXIS-UTIB-XXXXXXXXXXX7756-TRANSFER',
  '0000424320347279',
  '30/08/24',
  '1,000.00',
  '',
  '101,004.42'],
 ['31/08/24',
  '.IMPSP2P423409352210#21/08/2024210824-MIR2524395700585',
  'MIR2524395700585',
  '31/08/24',
  '4.14',
  '',
  '101,000.28'],
 ['31/08/24',
  'NEFTDR-UTIB0001355-SOURAVAXIS-NETBANK,MUM-N244243236619002-SEPTEXP',
  'N244243236619002',
  '31/08/24',
  '30,000.00',
  '',
  '71,000.28'],
 ['02/09/24',
  'NEFTDR-PUNB0017720-RICHAHORE-NETBANK,MUM-N246243238840545-TRANSFERRIC',
  'N246243238840545',
  '02/09/24',
  '4,765.00',
  '',
  '66,235.28'],
 ['02/09/24',
  'MEDCSI435584XXXXXX8577GOOGLEPLAYSERCYBSS',
  '0000424618050280',
  '03/09/24',
  '130.00',
  '',
  '66,105.28'],
 ['03/09/24',
  '.IMPSP2P423714320862#24/08/2024240824-MIR2524598746529',
  'MIR2524598746529',
  '03/09/24',
  '5.90',
  '',


In [174]:
pd.DataFrame(cleaned_list)

,0,1,2,3,4,5,6
0,30/08/24,082024SALARYCREDITFOR20532356/SOURAV,0000408297529599,30/08/24,,"74,941.00","102,004.42"
1,30/08/24,IMPS-424320347279-SOURAVAXIS-UTIB-XXXXXXXXXXX7...,0000424320347279,30/08/24,"1,000.00",,"101,004.42"
2,31/08/24,.IMPSP2P423409352210#21/08/2024210824-MIR25243...,MIR2524395700585,31/08/24,4.14,,"101,000.28"
3,31/08/24,"NEFTDR-UTIB0001355-SOURAVAXIS-NETBANK,MUM-N244...",N244243236619002,31/08/24,"30,000.00",,"71,000.28"
4,02/09/24,"NEFTDR-PUNB0017720-RICHAHORE-NETBANK,MUM-N2462...",N246243238840545,02/09/24,"4,765.00",,"66,235.28"
5,02/09/24,MEDCSI435584XXXXXX8577GOOGLEPLAYSERCYBSS,0000424618050280,03/09/24,130.00,,"66,105.28"
6,03/09/24,.IMPSP2P423714320862#24/08/2024240824-MIR25245...,MIR2524598746529,03/09/24,5.90,,"66,099.38"
7,03/09/24,.IMPSP2P423821370132#25/08/2024250824-MIR25246...,MIR2524699696806,03/09/24,4.14,,"66,095.24"
8,03/09/24,.IMPSP2P423921329460#26/08/2024260824-MIR25246...,MIR2524600400182,03/09/24,4.14,,"66,091.10"
9,04/09/24,ATW-435584XXXXXX8577-P3ENBL81-BANGALORENEFTDR-...,0000000000002921,04/09/24,"3,000.00",,"63,091.10"


In [26]:
""" Find the IFSC CODE """
input_file = r'input/Acct Statement_XX4725_29092024.pdf'
# input_file = r'input/statement_.pdf'
with contextlib.redirect_stderr(io.StringIO()):
    with pdfplumber.open(input_file) as pdf:
        page = pdf.pages[0]
        table = page.extract_text()
        # print('Extracted text:', table)

import re
pattern = r'[A-Z]{4}0[A-Z0-9]{6}'
match = re.search(pattern, table, re.IGNORECASE)
if match:
    ifsc_code = match.group()
    print("Found IFSC Code:", ifsc_code)
else:
    print("No IFSC Code found.")

    

Found IFSC Code: HDFC0004216


In [27]:
# Get Bank Name from IFSC Code
import requests
def get_bank_name(ifsc_code):
    url = f"https://ifsc.razorpay.com/{ifsc_code}"
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise error for bad status
        data = response.json()
        return data.get('BANK')
    except requests.exceptions.HTTPError:
        print("Invalid IFSC code or API error.")
        return None
    except Exception as e:
        print("Error:", e)
        return None

# Example usage
ifsc = "HDFC0004216"
bank_name = get_bank_name(ifsc)
if bank_name:
    print(f"Bank Name: {bank_name}")
else:
    print("Bank name not found.")


Bank Name: HDFC Bank
